# MapLibre Visualization

This notebook demonstrates GeoAgent's visualization capabilities using leafmap's MapLibre backend.

In [ ]:
# %pip install geoagent

## Create a Basic Map

In [ ]:
from leafmap.maplibregl import Map

m = Map(center=[-122.4194, 37.7749], zoom=10)
m

## Add a COG Layer

Load a Cloud-Optimized GeoTIFF directly on the map:

In [ ]:
m = Map(center=[-122.4194, 37.7749], zoom=10)

url = (
    "https://github.com/opengeos/datasets/releases/download/raster/Libya-2023-07-01.tif"
)
m.add_cog_layer(url, name="Libya Satellite")
m

## Add GeoJSON Data

In [ ]:
m = Map(center=[-100, 40], zoom=3)

geojson_url = "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_110m_admin_1_states_provinces_lines.geojson"
m.add_geojson(geojson_url, name="US States")
m

## Add PMTiles Layer

In [ ]:
m = Map(center=[0, 20], zoom=2)

pmtiles_url = "https://pmtiles.io/protomaps(vector)ODbL_firenze.pmtiles"
# m.add_pmtiles(pmtiles_url, name="Firenze")
m

## STAC Search and Visualization

Search for data and display it on a map:

In [ ]:
from geoagent.catalogs.registry import CatalogRegistry

reg = CatalogRegistry()
client = reg.get_client("earth_search")

results = client.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.5, 37.5, -122.0, 38.0],
    datetime="2024-07-01/2024-07-31",
    max_items=1,
)

items = list(results.items())
if items:
    item = items[0]
    print(f"Item: {item.id}")
    print(f"Visual asset: {item.assets.get('visual', {})}")

In [ ]:
if items and "visual" in item.assets:
    m = Map(center=[-122.25, 37.75], zoom=10)
    m.add_cog_layer(item.assets["visual"].href, name=item.id)
    m

---

## Drive the same map with GeoAgent

Building a `GeoAgent` with `map_obj=m` in its `GeoAgentContext` activates the mapping subagent and binds its tools (add layer, set basemap, fly to, save) to the live `Map` widget. Subsequent `agent.chat(...)` calls mutate `m` in place, so re-displaying the widget shows the agent's work.

Prefer this over `agent.chat(query, target_map=m)`, which rebuilds the deepagents graph and rotates the LangGraph thread id (resetting conversation history). The lower-level `for_leafmap(m)` factory is also available, but it returns a raw deepagents graph rather than the `GeoAgent` facade — see [live_mapping.ipynb](live_mapping.ipynb) for a side-by-side comparison.


In [ ]:
from geoagent import GeoAgent, GeoAgentContext
from leafmap.maplibregl import Map

live_map = Map(center=[-122.4194, 37.7749], zoom=10)
agent = GeoAgent(context=GeoAgentContext(map_obj=live_map))

In [ ]:
result = agent.chat("Add a Sentinel-2 RGB layer over San Francisco for July 2024.")
print("Tools fired:", result.executed_tools)
live_map

In [ ]:
result = agent.chat("List the current map layers.")
print("Tools fired:", result.executed_tools)
if result.answer_text:
    print(result.answer_text[:500])